# Notebook 15: Exporting Fine-tuned Models

## Overview

- **Duration**: ~1 hour
- **Prerequisites**: Notebook 13-14 (Fine-tuning and Evaluation)
- **Learning Objectives**:
  1. Merge LoRA weights into base model
  2. Export to GGUF format for llama.cpp
  3. Push model to Hugging Face Hub
  4. Understand different export options

## Introduction

### Why Export?

After training, you may want to:
- **Deploy** the model in production
- **Share** with others via Hugging Face Hub
- **Run locally** using tools like llama.cpp or Ollama
- **Reduce size** through quantization

### Export Formats

| Format | Use Case | Size |
|--------|----------|------|
| LoRA adapter | Continue training, small storage | ~50MB |
| Merged FP16 | HuggingFace inference | Full size |
| GGUF Q4_K_M | llama.cpp, Ollama | ~40% of FP16 |
| GGUF Q8_0 | Higher quality local | ~50% of FP16 |

In [ ]:
import torch
import os
from unsloth import FastLanguageModel

print(f"CUDA available: {torch.cuda.is_available()}")

## Step 1: Load the Fine-tuned Model

Load your trained model with LoRA adapters.

In [ ]:
# Load base model
model_name = "unsloth/Llama-3.2-3B-bnb-4bit"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Load LoRA adapter
from peft import PeftModel
model = PeftModel.from_pretrained(model, "my_lora_adapter")

print("Model loaded with LoRA adapter!")

## Exercise 15.1: Save Merged Model (20 min)

Merge LoRA weights into the base model and save in different formats.

In [ ]:
# Output directory
output_dir = "my_merged_model"

# TODO: Save merged model in 16-bit format
# Unsloth provides save_pretrained_merged() for this
#
model.save_pretrained_merged(
    output_dir,
    tokenizer,
    save_method="merged_16bit",  # Options: merged_16bit, merged_4bit, lora
)

raise NotImplementedError("Save merged model")

In [ ]:
# Verify saved files
if os.path.exists(output_dir):
    total_size = 0
    print(f"Files in {output_dir}:")
    for f in sorted(os.listdir(output_dir)):
        path = os.path.join(output_dir, f)
        size = os.path.getsize(path) / 1e9  # GB
        total_size += size
        print(f"  {f}: {size:.2f} GB")
    print(f"\nTotal: {total_size:.2f} GB")

## Exercise 15.2: Export to GGUF (20 min)

GGUF is the format used by llama.cpp and Ollama for efficient local inference.

### Quantization Options

| Method | Bits | Quality | Size |
|--------|------|---------|------|
| q4_k_m | 4 | Good | ~40% |
| q5_k_m | 5 | Better | ~45% |
| q8_0 | 8 | Best | ~50% |
| f16 | 16 | Original | 100% |

In [ ]:
# TODO: Export to GGUF format
#
# Unsloth can export directly to GGUF:
model.save_pretrained_gguf(
    "my_model_gguf",
    tokenizer,
    quantization_method="q4_k_m",  # Choose quantization
)

# Available methods: q4_k_m, q5_k_m, q8_0, f16

raise NotImplementedError("Export to GGUF")

In [ ]:
# Alternative: Export multiple quantization levels
quantization_methods = ["q4_k_m", "q8_0"]

for method in quantization_methods:
    output_name = f"my_model_{method}"
    print(f"\nExporting {method}...")
    # model.save_pretrained_gguf(output_name, tokenizer, quantization_method=method)

## Exercise 15.3: Push to Hugging Face Hub (15 min)

Share your model with the community!

In [ ]:
# First, login to Hugging Face
# Option 1: Run this in terminal
# !huggingface-cli login

# Option 2: Use token directly
from huggingface_hub import login
login(token="your_token_here")

In [ ]:
# TODO: Push to Hugging Face Hub
#
model.push_to_hub_merged(
    "your-username/your-model-name",
    tokenizer,
    save_method="merged_16bit",
    token="your_token",  # Optional if logged in
)
#
# Or push just the LoRA adapter:
model.push_to_hub(
    "your-username/your-lora-adapter",
    token="your_token",
)

raise NotImplementedError("Push to Hub")

In [ ]:
# Push GGUF to Hub
model.push_to_hub_gguf(
    "your-username/your-model-gguf",
    tokenizer,
    quantization_method="q4_k_m",
    token="your_token",
)

## Using Your Exported Model

### With llama.cpp

```bash
# Install llama.cpp
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp && make

# Run inference
./llama-cli -m my_model_q4_k_m.gguf -p "What is machine learning?" -n 256
```

### With Ollama

```bash
# Create a Modelfile
echo 'FROM ./my_model_q4_k_m.gguf' > Modelfile

# Create Ollama model
ollama create my-model -f Modelfile

# Run
ollama run my-model
```

### Load from Hugging Face

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("your-username/your-model-name")
tokenizer = AutoTokenizer.from_pretrained("your-username/your-model-name")
```

## Exercise 15.4: Create Model Card (10 min)

A good model card helps others understand and use your model.

In [ ]:
model_card_template = """
---
language:
- en
license: apache-2.0
tags:
- llama
- fine-tuned
- lora
base_model: meta-llama/Llama-3.2-3B
---

# My Fine-tuned Llama 3.2 Model

## Model Description

This model is a fine-tuned version of Llama 3.2 3B, trained on instruction-following data.

## Training Details

- **Base Model**: meta-llama/Llama-3.2-3B
- **Training Data**: Alpaca Cleaned (~52k examples)
- **LoRA Rank**: 16
- **Learning Rate**: 2e-4
- **Epochs**: 1
- **Hardware**: [Your GPU]

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("your-username/your-model")
tokenizer = AutoTokenizer.from_pretrained("your-username/your-model")

prompt = "What is machine learning?"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0]))
```

## Limitations

- Fine-tuned on English data only
- May hallucinate or generate incorrect information
- Should not be used for medical, legal, or financial advice

## License

This model is released under the Apache 2.0 license.
"""

print(model_card_template)

In [ ]:
# TODO: Save the model card
#
with open(os.path.join(output_dir, "README.md"), "w") as f:
    f.write(model_card_template)
#
print(f"Model card saved to {output_dir}/README.md")

raise NotImplementedError("Save model card")

## Summary

### What You Learned

1. **Merging Weights**: Combine LoRA adapters with base model
2. **GGUF Export**: Create quantized models for local inference
3. **Hub Upload**: Share models on Hugging Face
4. **Model Cards**: Document your model properly

### Export Recommendations

| Goal | Format | Method |
|------|--------|--------|
| Continue training | LoRA adapter | `save_pretrained()` |
| HF inference | Merged FP16 | `save_pretrained_merged()` |
| Local CPU/GPU | GGUF q4_k_m | `save_pretrained_gguf()` |
| Best local quality | GGUF q8_0 | `save_pretrained_gguf()` |

### Course Complete!

Congratulations! You've learned how to:
- Build a GPT from scratch (Week 1)
- Fine-tune modern LLMs with LoRA (Week 2)
- Export and deploy your models

Keep experimenting and building!